In [1]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Common Spark Types ")
    .config("spark.master", "local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/21 15:04:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
moviesDF = spark.read.json("src/main/resources/data/movies.json")

moviesDF.show(5)

+--------------------+--------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|       Creative_Type|Director|Distributor|IMDB_Rating|IMDB_Votes|MPAA_Rating|Major_Genre|Production_Budget|Release_Date|Rotten_Tomatoes_Rating|Running_Time_min|             Source|               Title|US_DVD_Sales|US_Gross|Worldwide_Gross|
+--------------------+--------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|                NULL|    NULL|   Gramercy|        6.1|      1071|          R|       NULL|          8000000|   12-Jun-98|                  NULL|            NULL|               NULL|      The Land Girls|        NULL|  146083|         146083|
|                NULL|    NULL|     

In [ ]:
# how to insert certain values into data frames?

# adding a plain value to a DF, the lit function will work across types
moviesDF.select(col("Title"), lit(47).alias("plain_value")).show(5)

+--------------------+-----------+
|               Title|plain_value|
+--------------------+-----------+
|      The Land Girls|         47|
|First Love, Last ...|         47|
|I Married a Stran...|         47|
|Let's Talk About Sex|         47|
|                Slam|         47|
+--------------------+-----------+
only showing top 5 rows



In [21]:
# booleans
# regular filtering
dramaFilter = col("Major_Genre") == "Drama"
goodRatingFilter = col("IMDB_Rating") > 7.0
preferredFilter = dramaFilter & goodRatingFilter
moviesDF.select("Title") \
.where(dramaFilter)

# add new boolean column
moviesWithGoodnessFlags = moviesDF.select(col("Title"), preferredFilter.alias("good_movie"))

moviesWithGoodnessFlags.show(5)

+--------------------+----------+
|               Title|good_movie|
+--------------------+----------+
|      The Land Girls|     false|
|First Love, Last ...|     false|
|I Married a Stran...|     false|
|Let's Talk About Sex|     false|
|                Slam|     false|
+--------------------+----------+
only showing top 5 rows



In [22]:
# filter on boolean column
moviesWithGoodnessFlags.where("good_movie").show(5)

+----------------+----------+
|           Title|good_movie|
+----------------+----------+
|    12 Angry Men|      true|
|  Twelve Monkeys|      true|
|Twin Falls Idaho|      true|
|            Amen|      true|
|    Barry Lyndon|      true|
+----------------+----------+
only showing top 5 rows



In [25]:
# negations
moviesWithGoodnessFlags.where(~col("good_movie")).show(5)

+--------------------+----------+
|               Title|good_movie|
+--------------------+----------+
|      The Land Girls|     false|
|First Love, Last ...|     false|
|I Married a Stran...|     false|
|Let's Talk About Sex|     false|
|                Slam|     false|
+--------------------+----------+
only showing top 5 rows



In [26]:
# numbers

# can use math operators on spark numerical cols
moviesAvgRatingsDF = moviesDF.select(col("Title"), (col("Rotten_Tomatoes_Rating") / 10 + col("IMDB_Rating")) / 2).show(5)

+--------------------+---------------------------------------------------+
|               Title|(((Rotten_Tomatoes_Rating / 10) + IMDB_Rating) / 2)|
+--------------------+---------------------------------------------------+
|      The Land Girls|                                               NULL|
|First Love, Last ...|                                               NULL|
|I Married a Stran...|                                               NULL|
|Let's Talk About Sex|                                               NULL|
|                Slam|                                                4.8|
+--------------------+---------------------------------------------------+
only showing top 5 rows



In [27]:
# correlation = number between -1 and 1
# -1 is negative correlation, 1 is closely correlated
# near 0 means little to no correlation
print(moviesDF.stat.corr("Rotten_Tomatoes_Rating", "IMDB_Rating")) # corr is an ACTION, so executes immediately

0.4259708986248317


In [ ]:
# strings processing
carsDF = spark.read.json("src/main/resources/data/cars.json")

# init cap, - capitalize first character in each word
# we also have lower and upper
carsDF.select(initcap("Name")).show(5)

+--------------------+
|       initcap(Name)|
+--------------------+
|Chevrolet Chevell...|
|   Buick Skylark 320|
|  Plymouth Satellite|
|       Amc Rebel Sst|
|         Ford Torino|
+--------------------+
only showing top 5 rows



In [31]:
# contains - check if string contains substring
carsDF.select("*").where(col("Name").contains("volkswagen")).show(5)

+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|Acceleration|Cylinders|Displacement|Horsepower|Miles_per_Gallon|                Name|Origin|Weight_in_lbs|      Year|
+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|        20.5|        4|        97.0|        46|            26.0|volkswagen 1131 d...|Europe|         1835|1970-01-01|
|        20.0|        4|        97.0|        48|            NULL|volkswagen super ...|Europe|         1978|1971-01-01|
|        19.0|        4|        97.0|        60|            27.0|volkswagen model 111|Europe|         1834|1971-01-01|
|        23.5|        4|        97.0|        54|            23.0|   volkswagen type 3|Europe|         2254|1972-01-01|
|        18.0|        4|       121.0|        76|            22.0| volkswagen 411 (sw)|Europe|         2511|1972-01-01|
+------------+---------+------------+----------+

In [34]:
# regex - stronger matching
regexString = "volkswagen|vw"
vwDF = carsDF.select("Name", regexp_extract(col("name"), regexString, 0).alias("regex_extract")).where(col("regex_extract") != "")

vwDF.show(5)

+--------------------+-------------+
|                Name|regex_extract|
+--------------------+-------------+
|volkswagen 1131 d...|   volkswagen|
|volkswagen super ...|   volkswagen|
|volkswagen model 111|   volkswagen|
|   volkswagen type 3|   volkswagen|
| volkswagen 411 (sw)|   volkswagen|
+--------------------+-------------+
only showing top 5 rows



In [ ]:
# regex to replace
(
    vwDF.select(
        col("Name"),
        regexp_replace(col("Name"), regexString, "People's Car")
    )
).show(5)

+--------------------+----------------------------------------------------+
|                Name|regexp_replace(Name, volkswagen|vw, People's Car, 1)|
+--------------------+----------------------------------------------------+
|volkswagen 1131 d...|                                People's Car 1131...|
|volkswagen super ...|                                People's Car supe...|
|volkswagen model 111|                                People's Car mode...|
|   volkswagen type 3|                                 People's Car type 3|
| volkswagen 411 (sw)|                                People's Car 411 ...|
+--------------------+----------------------------------------------------+
only showing top 5 rows



In [ ]:
# exercise filter cars df with a list of car names
# someone gives me api call defGetCarNames which returns a list of strings
# but once you get the car names create a filter with the cars no matter what car names are in the list

In [41]:
# transfom array to regex
dynamicCarNames = ["amc hornet", "ford"]

regexString = '|'.join(dynamicCarNames)


In [42]:

carsDF.select("Name", regexp_extract(col("name"), regexString, 0).alias("regex_extract")).where(col("regex_extract") != "").show(5)


+--------------------+-------------+
|                Name|regex_extract|
+--------------------+-------------+
|         ford torino|         ford|
|    ford galaxie 500|         ford|
|    ford torino (sw)|         ford|
|ford mustang boss...|         ford|
|          amc hornet|   amc hornet|
+--------------------+-------------+
only showing top 5 rows

